In [ ]:
import os
import sys
import subprocess

PY = sys.executable
BIN_DIR = os.path.dirname(PY)
CNMF = os.path.join(BIN_DIR, "cnmf")

print("Python executable:", PY)
print("Bin dir:", BIN_DIR)
print("cNMF executable:", CNMF)

BASE_DIR = "/.../.../"
INPUT_GLOBAL_H5AD = "/.../.../Global.h5ad"

os.makedirs(BASE_DIR, exist_ok=True)

INPUT_B_H5AD = os.path.join(BASE_DIR, "Bcells_allstates.h5ad")
OUTDIR = os.path.join(BASE_DIR, "cnmf_out")
NAME = "Bcells_allstates"

os.makedirs(OUTDIR, exist_ok=True)

# sanity check
subprocess.check_call([CNMF, "--help"])

In [ ]:
import scanpy as sc
import numpy as np
from scipy import sparse

adata = sc.read_h5ad(INPUT_GLOBAL_H5AD)
print(adata)

# inspect available labels first
print(sorted(adata.obs["Cluster_celltypes"].astype(str).unique())[:50])

In [ ]:
B_label = "B cells"

adata_B = adata[adata.obs["Cluster_celltypes"].astype(str) == B_label].copy()
print(adata_B)

# confirm both states are present
print(adata_B.obs["Stimulation"].value_counts(dropna=False))
print(adata_B.obs["Lifestyle"].value_counts(dropna=False))

In [ ]:
# ensure sparse CSR
if not sparse.issparse(adata_B.X):
    adata_B.X = sparse.csr_matrix(adata_B.X)
else:
    adata_B.X = adata_B.X.tocsr()

cell_sums = np.array(adata_B.X.sum(axis=1)).ravel()
gene_sums = np.array(adata_B.X.sum(axis=0)).ravel()

print("Cells with zero counts:", (cell_sums == 0).sum())
print("Genes with zero counts:", (gene_sums == 0).sum())

adata_B = adata_B[cell_sums > 0, gene_sums > 0].copy()
print("After safety filtering:", adata_B)

adata_B.write_h5ad(INPUT_B_H5AD)
print("Saved:", INPUT_B_H5AD)

In [ ]:
KS = [4, 5, 6, 7, 8]
N_ITER = 50
NUMGENES = 2000
SEED = 14
TOTAL_WORKERS = 1

print("K values:", KS)
print("n_iter:", N_ITER)
print("numgenes:", NUMGENES)

In [ ]:
prepare_cmd = [
    CNMF, "prepare",
    "--output-dir", OUTDIR,
    "--name", NAME,
    "-c", INPUT_B_H5AD,
    "-k", *map(str, KS),
    "--n-iter", str(N_ITER),
    "--seed", str(SEED),
    "--numgenes", str(NUMGENES),
    "--total-workers", str(TOTAL_WORKERS),
]

print("Running:", " ".join(prepare_cmd))
subprocess.check_call(prepare_cmd)

In [ ]:
factorize_cmd = [
    CNMF, "factorize",
    "--output-dir", OUTDIR,
    "--name", NAME,
    "--worker-index", "0",
    "--total-workers", str(TOTAL_WORKERS),
]

print("Running:", " ".join(factorize_cmd))
subprocess.check_call(factorize_cmd)

In [ ]:
combine_cmd = [
    CNMF, "combine",
    "--output-dir", OUTDIR,
    "--name", NAME,
]

print("Running:", " ".join(combine_cmd))
subprocess.check_call(combine_cmd)

In [ ]:
kplot_cmd = [
    CNMF, "k_selection_plot",
    "--output-dir", OUTDIR,
    "--name", NAME,
]

print("Running:", " ".join(kplot_cmd))
subprocess.check_call(kplot_cmd)

In [ ]:
from IPython.display import Image, display

kplot_file = os.path.join(OUTDIR, NAME, f"{NAME}.k_selection.png")
print(kplot_file)
display(Image(filename=kplot_file))

In [ ]:
CHOSEN_K = 6
DENSITY_THRESHOLD = 0.1

consensus_cmd = [
    CNMF, "consensus",
    "--output-dir", OUTDIR,
    "--name", NAME,
    "--components", str(CHOSEN_K),
    "--local-density-threshold", str(DENSITY_THRESHOLD),
    "--show-clustering",
]

print("Running:", " ".join(consensus_cmd))
subprocess.check_call(consensus_cmd)

In [ ]:
import os

run_dir = os.path.join(OUTDIR, NAME)
for f in sorted(os.listdir(run_dir)):
    print(f)

In [ ]:
from cnmf import cNMF

cnmf_obj = cNMF(output_dir=OUTDIR, name=NAME)

usage_bcells, spectra_scores_bcells, spectra_tpm_bcells, top_genes_bcells = cnmf_obj.load_results(
    K=CHOSEN_K,
    density_threshold=0.1
)

print("usage:", usage_bcells.shape)
print("spectra_scores:", spectra_scores_bcells.shape)
print("spectra_tpm:", spectra_tpm_bcells.shape)

In [ ]:
adata_B = sc.read_h5ad(INPUT_B_H5AD)

usage_bcells = usage_bcells.reindex(adata_B.obs_names)
usage_bcells.columns = [f"cNMF_bcells_{c}" for c in usage_bcells.columns]

adata_B.obs[usage_bcells.columns] = usage_bcells
program_cols = usage_bcells.columns.tolist()

print(program_cols)

In [ ]:
import pandas as pd

umap_df = pd.read_csv(
    "/.../.../seurat_bcells_umap.csv"
).set_index("cell_id")

umap_df = umap_df.loc[adata_B.obs_names]
adata_B.obsm["X_umap"] = umap_df.iloc[:, :2].to_numpy()

In [ ]:
TOP_N = 25

for prog in spectra_scores_bcells.columns:
    print(f"\n=== Program {prog} ===")
    top = spectra_scores_bcells[prog].sort_values(ascending=False).head(TOP_N)
    print(", ".join(top.index.tolist()))

In [ ]:
print(len(adata.obs_names))
print(len(umap_df.index))
print(sum(adata.obs_names.isin(umap_df.index)))

In [ ]:
sc.pl.umap(adata_B, color=program_cols, ncols=3)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

stim_means = adata_B.obs.groupby("Stimulation", observed=False)[program_cols].mean()
life_means = adata_B.obs.groupby("Lifestyle", observed=False)[program_cols].mean()
stim_life_means = adata_B.obs.groupby(["Stimulation", "Lifestyle"], observed=False)[program_cols].mean()

plt.figure(figsize=(10, 4))
sns.heatmap(stim_means, cmap="viridis")
plt.title("B cells cNMF program usage by stimulation")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 4))
sns.heatmap(life_means, cmap="viridis")
plt.title("B cells cNMF program usage by lifestyle")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 6))
sns.heatmap(stim_life_means, cmap="viridis")
plt.title("B cells cNMF program usage by stimulation and lifestyle")
plt.tight_layout()
plt.show()

In [ ]:
df_plot = adata_B.obs[["Stimulation", "Lifestyle"] + program_cols].copy()

df_long = df_plot.melt(
    id_vars=["Stimulation", "Lifestyle"],
    value_vars=program_cols,
    var_name="Program",
    value_name="Usage"
)

g = sns.catplot(
    data=df_long,
    x="Stimulation",
    y="Usage",
    hue="Lifestyle",
    col="Program",
    kind="box",
    col_wrap=3,
    sharey=True,
    height=3,
    showfliers=False
)

g.fig.suptitle("B cells cNMF program usage — Lifestyle × Stimulation", y=1.02)
plt.show()

In [ ]:
usage_bcells.to_csv(os.path.join(BASE_DIR, f"{NAME}.usage.k{CHOSEN_K}.dt0_1.csv"))
spectra_scores_bcells.to_csv(os.path.join(BASE_DIR, f"{NAME}.spectra_scores.k{CHOSEN_K}.dt0_1.csv"))
spectra_tpm_bcells.to_csv(os.path.join(BASE_DIR, f"{NAME}.spectra_tpm.k{CHOSEN_K}.dt0_1.csv"))

adata_B.write_h5ad(os.path.join(BASE_DIR, f"{NAME}.with_cNMF.h5ad"))

In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# ------------------------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------------------------
sample_col = "library"   # change to "library" if you want library instead
outcome_var = "Usage_log1p"   # or "Usage"

program_cols = [c for c in adata_B.obs.columns if c.startswith("cNMF_")]

required_cols = [sample_col, "Stimulation", "Lifestyle", "Age", "Sex"] + program_cols
missing = [c for c in required_cols if c not in adata_B.obs.columns]
if missing:
    raise ValueError(f"Missing required columns in adata_B.obs: {missing}")

# ------------------------------------------------------------------------------
# 1) CELL-LEVEL LONG TABLE
# ------------------------------------------------------------------------------
df_cells = adata_B.obs[required_cols].copy()

df_long = df_cells.melt(
    id_vars=[sample_col, "Stimulation", "Lifestyle", "Age", "Sex"],
    value_vars=program_cols,
    var_name="Program",
    value_name="Usage"
)

df_long = df_long.dropna(
    subset=[sample_col, "Stimulation", "Lifestyle", "Age", "Sex", "Program", "Usage"]
).copy()

# Important: make Program plain string, not categorical
df_long["Program"] = df_long["Program"].astype(str)
df_long["Lifestyle"] = df_long["Lifestyle"].astype(str)
df_long["Stimulation"] = df_long["Stimulation"].astype(str)
df_long["Sex"] = df_long["Sex"].astype(str)

# ------------------------------------------------------------------------------
# 2) AGGREGATE TO SAMPLE/DONOR LEVEL
# ------------------------------------------------------------------------------
df_donor = (
    df_long
    .groupby([sample_col, "Stimulation", "Lifestyle", "Program"], observed=True)
    .agg(
        Usage=("Usage", "mean"),
        Age=("Age", "first"),
        Sex=("Sex", "first"),
        n_cells=("Usage", "size")
    )
    .reset_index()
)

print(df_donor.head())
print(df_donor.shape)

# ------------------------------------------------------------------------------
# 3) FORMAT VARIABLES
# ------------------------------------------------------------------------------
df_donor["Age"] = pd.to_numeric(df_donor["Age"], errors="coerce")
df_donor = df_donor.dropna(subset=["Age"]).copy()

df_donor["Age_z"] = (df_donor["Age"] - df_donor["Age"].mean()) / df_donor["Age"].std(ddof=0)
df_donor["Usage_log1p"] = np.log1p(df_donor["Usage"])

# ------------------------------------------------------------------------------
# 4) LINEAR MODEL WITHIN EACH PROGRAM x STIMULATION
#    Urban vs Rural adjusted for Age + Sex
# ------------------------------------------------------------------------------
results = []

for program in sorted(df_donor["Program"].unique()):
    for stim in sorted(df_donor["Stimulation"].unique()):
        sub = df_donor[
            (df_donor["Program"] == program) &
            (df_donor["Stimulation"] == stim)
        ].copy()

        sub = sub.dropna(subset=[outcome_var, "Lifestyle", "Age_z", "Sex"])

        # Require both groups
        lifestyles_present = set(sub["Lifestyle"].unique())
        if not {"RURAL", "URBAN"}.issubset(lifestyles_present):
            print(f"Skipping {program} / {stim}: missing one lifestyle group")
            continue

        # Set reference levels
        sub["Lifestyle"] = pd.Categorical(sub["Lifestyle"], categories=["RURAL", "URBAN"])
        sex_levels = sorted(sub["Sex"].unique())
        sub["Sex"] = pd.Categorical(sub["Sex"], categories=sex_levels)

        if sub.shape[0] < 6:
            print(f"Skipping {program} / {stim}: too few samples ({sub.shape[0]})")
            continue

        formula = f"{outcome_var} ~ C(Lifestyle) + Age_z + C(Sex)"
        model = smf.ols(formula, data=sub).fit()

        term = "C(Lifestyle)[T.URBAN]"
        if term not in model.params.index:
            print(f"Skipping {program} / {stim}: lifestyle coefficient not found")
            continue

        results.append({
            "Program": program,
            "Stimulation": stim,
            "n_samples": sub.shape[0],
            "n_rural": (sub["Lifestyle"].astype(str) == "RURAL").sum(),
            "n_urban": (sub["Lifestyle"].astype(str) == "URBAN").sum(),
            "coef_urban_vs_rural": model.params[term],
            "se": model.bse[term],
            "t": model.tvalues[term],
            "p": model.pvalues[term],
            "ci_low": model.conf_int().loc[term, 0],
            "ci_high": model.conf_int().loc[term, 1],
            "r2": model.rsquared,
            "adj_r2": model.rsquared_adj,
            "mean_rural": sub.loc[sub["Lifestyle"].astype(str) == "RURAL", "Usage"].mean(),
            "mean_urban": sub.loc[sub["Lifestyle"].astype(str) == "URBAN", "Usage"].mean(),
            "median_rural": sub.loc[sub["Lifestyle"].astype(str) == "RURAL", "Usage"].median(),
            "median_urban": sub.loc[sub["Lifestyle"].astype(str) == "URBAN", "Usage"].median(),
        })

res_df = pd.DataFrame(results)

if not res_df.empty:
    res_df["p_adj_BH"] = multipletests(res_df["p"], method="fdr_bh")[1]
    res_df["significant_BH_0.05"] = res_df["p_adj_BH"] < 0.05

res_df = res_df.sort_values(["Program", "Stimulation"]).reset_index(drop=True)

print(res_df)
# res_df.to_csv("program_usage_lm_by_program_and_stimulation.csv", index=False)

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# =========================================================
# 0. SETTINGS
# =========================================================
adata_obj = adata_B   # change for other cell types
sample_col = "library"   # or donor_id
stim_col = "Stimulation"
life_col = "Lifestyle"
sex_col = "Sex"
age_col = "Age"

life_levels = ["RURAL", "URBAN"]
sex_levels = ["Female", "Male"]

program_cols = [c for c in adata_obj.obs.columns if c.startswith("cNMF_")]
if len(program_cols) == 0:
    raise ValueError("No cNMF program columns found.")

required_cols = [sample_col, stim_col, life_col, sex_col, age_col] + program_cols
missing = [c for c in required_cols if c not in adata_obj.obs.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

sns.set(style="whitegrid", context="notebook")

# =========================================================
# 1. SINGLE-CELL LONG TABLE (FOR PLOTTING)
# =========================================================
df_plot = adata_obj.obs[required_cols].copy()

df_long = df_plot.melt(
    id_vars=[sample_col, stim_col, life_col, sex_col, age_col],
    value_vars=program_cols,
    var_name="Program",
    value_name="Usage"
).dropna().copy()

df_long["Program"] = df_long["Program"].astype(str)
df_long[stim_col] = df_long[stim_col].astype(str)
df_long[life_col] = df_long[life_col].astype(str)
df_long[sex_col] = df_long[sex_col].astype(str)
df_long[age_col] = pd.to_numeric(df_long[age_col], errors="coerce")
df_long = df_long.dropna(subset=[age_col]).copy()

print("Single-cell table shape:", df_long.shape)
print(df_long.head())

# =========================================================
# 2. DONOR-/LIBRARY-LEVEL TABLE (FOR MODELING)
# =========================================================
df_donor = (
    df_long
    .groupby([sample_col, stim_col, life_col, sex_col, "Program"], observed=True)
    .agg(
        Usage=("Usage", "mean"),
        Age=(age_col, "first"),
        n_cells=("Usage", "size")
    )
    .reset_index()
)

df_donor["Age"] = pd.to_numeric(df_donor["Age"], errors="coerce")
df_donor = df_donor.dropna(subset=["Age"]).copy()

# same transformed outcome as before
df_donor["Usage_log1p"] = np.log1p(df_donor["Usage"])
df_donor["Age_z"] = (df_donor["Age"] - df_donor["Age"].mean()) / df_donor["Age"].std(ddof=0)

df_donor[life_col] = df_donor[life_col].astype(str)
df_donor[sex_col] = df_donor[sex_col].astype(str)
df_donor[stim_col] = df_donor[stim_col].astype(str)

print("Donor-level table shape:", df_donor.shape)
print(df_donor.head())

# =========================================================
# 3. FIT ORIGINAL WITHIN-STIMULATION MODEL
#    Usage_log1p ~ Lifestyle + Age_z + Sex
#
#    One model per Program x Stimulation
#    Coefficient of interest = Urban vs Rural
# =========================================================
results = []

for program in sorted(df_donor["Program"].unique()):
    for stim in sorted(df_donor[stim_col].unique()):

        sub = df_donor[
            (df_donor["Program"] == program) &
            (df_donor[stim_col] == stim)
        ].copy()

        sub = sub.dropna(subset=["Usage_log1p", "Age_z", life_col, sex_col])

        lifestyles_present = set(sub[life_col].unique())
        sexes_present = set(sub[sex_col].unique())

        if not set(life_levels).issubset(lifestyles_present):
            continue
        if not set(sex_levels).issubset(sexes_present):
            continue
        if sub.shape[0] < 6:
            continue

        sub[life_col] = pd.Categorical(sub[life_col], categories=life_levels, ordered=True)
        sub[sex_col] = pd.Categorical(sub[sex_col], categories=sex_levels, ordered=True)

        model = smf.ols(
            "Usage_log1p ~ C(Lifestyle) + Age_z + C(Sex)",
            data=sub
        ).fit()

        term = "C(Lifestyle)[T.URBAN]"
        if term not in model.params.index:
            continue

        results.append({
            "Program": program,
            "Stimulation": stim,
            "n_samples": sub.shape[0],
            "coef_urban_vs_rural": model.params[term],
            "se": model.bse[term],
            "t": model.tvalues[term],
            "p": model.pvalues[term],
            "ci_low": model.conf_int().loc[term, 0],
            "ci_high": model.conf_int().loc[term, 1],
            "r2": model.rsquared,
            "adj_r2": model.rsquared_adj,
            "mean_rural": sub.loc[sub[life_col] == "RURAL", "Usage"].mean(),
            "mean_urban": sub.loc[sub[life_col] == "URBAN", "Usage"].mean(),
            "median_rural": sub.loc[sub[life_col] == "RURAL", "Usage"].median(),
            "median_urban": sub.loc[sub[life_col] == "URBAN", "Usage"].median(),
        })

res_main = pd.DataFrame(results)

if not res_main.empty:
    res_main["p_adj_BH"] = multipletests(res_main["p"], method="fdr_bh")[1]
    res_main["significant_BH_0.05"] = res_main["p_adj_BH"] < 0.05

print("\nWithin-stimulation Urban vs Rural results:")
print(res_main.sort_values(["Program", "Stimulation"]))

# =========================================================
# 4. HELPER FOR LABELS
# =========================================================
def format_p(p):
    if pd.isna(p):
        return ""
    if p < 1e-3:
        return f"FDR={p:.1e}"
    return f"FDR={p:.3f}"

# =========================================================
# 5. SINGLE-CELL BOXPLOTS WITH DONOR-LEVEL ADJUSTED P-VALUES
#    x = Stimulation
#    hue = Lifestyle
#    panels = Programs
#
#    We use matplotlib subplots directly so layout is stable
# =========================================================
program_order = sorted(df_long["Program"].unique())
stim_order = sorted(df_long[stim_col].unique())

n_programs = len(program_order)
n_cols = 3
n_rows = int(np.ceil(n_programs / n_cols))

fig, axes = plt.subplots(
    n_rows, n_cols,
    figsize=(4.4 * n_cols, 3.8 * n_rows),
    squeeze=False,
    sharey=False
)

axes_flat = axes.flatten()

for idx, program in enumerate(program_order):
    ax = axes_flat[idx]

    sub_plot = df_long[df_long["Program"] == program].copy()
    if sub_plot.empty:
        ax.set_visible(False)
        continue

    sns.boxplot(
        data=sub_plot,
        x=stim_col,
        y="Usage",
        hue=life_col,
        ax=ax,
        order=stim_order,
        hue_order=life_levels,
        showfliers=False
    )

    # remove repeated legends from each panel
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    ymax = sub_plot["Usage"].max()
    ymin = sub_plot["Usage"].min()
    yrange = ymax - ymin if ymax > ymin else 0.05

    for i, stim in enumerate(stim_order):
        row = res_main[
            (res_main["Program"] == program) &
            (res_main["Stimulation"] == stim)
        ]
        if row.empty:
            continue

        label = format_p(row["p_adj_BH"].iloc[0])

        ax.text(
            i,
            ymax + 0.08 * yrange,
            label,
            ha="center",
            va="bottom",
            fontsize=8
        )

    ax.set_ylim(ymin, ymax + 0.22 * yrange)
    ax.set_title(program)
    ax.set_xlabel("")
    ax.set_ylabel("Usage")

# hide unused axes
for j in range(n_programs, len(axes_flat)):
    axes_flat[j].set_visible(False)

# one shared legend
handles, labels = axes_flat[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    title="Lifestyle",
    loc="upper center",
    ncol=2,
    bbox_to_anchor=(0.5, 1.02)
)

fig.suptitle("Single-cell program usage — Lifestyle × Stimulation", y=1.08)
plt.tight_layout()
plt.show()

# =========================================================
# 6. OPTIONAL SAVE
# =========================================================
# res_main.to_csv("within_stimulation_urban_vs_rural_lm_results.csv", index=False)

In [ ]:
import math
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

# =========================================================
# 0. SETTINGS
# =========================================================
adata_obj = adata_B          # change to other AnnData object as needed
sample_col = "library"          # or donor_id
stim_col = "Stimulation"
life_col = "Lifestyle"
sex_col = "Sex"
age_col = "Age"

# exact label levels expected in metadata
life_levels = ["RURAL", "URBAN"]
sex_levels = ["Female", "Male"]

# detect cNMF program columns
program_cols = [c for c in adata_obj.obs.columns if c.startswith("cNMF_")]
if len(program_cols) == 0:
    raise ValueError("No cNMF program columns found.")

required_cols = [sample_col, stim_col, life_col, sex_col, age_col] + program_cols
missing = [c for c in required_cols if c not in adata_obj.obs.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

# plotting style
sns.set(style="whitegrid", context="notebook")

# =========================================================
# 1. BUILD SINGLE-CELL LONG TABLE (FOR PLOTTING)
# =========================================================
df_plot = adata_obj.obs[required_cols].copy()

df_long = df_plot.melt(
    id_vars=[sample_col, stim_col, life_col, sex_col, age_col],
    value_vars=program_cols,
    var_name="Program",
    value_name="Usage"
).dropna().copy()

df_long["Program"] = df_long["Program"].astype(str)
df_long[stim_col] = df_long[stim_col].astype(str)
df_long[life_col] = df_long[life_col].astype(str)
df_long[sex_col] = df_long[sex_col].astype(str)
df_long[age_col] = pd.to_numeric(df_long[age_col], errors="coerce")
df_long = df_long.dropna(subset=[age_col]).copy()

print("Single-cell table shape:", df_long.shape)
print(df_long.head())

# =========================================================
# 2. BUILD DONOR-/LIBRARY-LEVEL TABLE (FOR MODELING)
# =========================================================
df_donor = (
    df_long
    .groupby([sample_col, stim_col, life_col, sex_col, "Program"], observed=True)
    .agg(
        Usage=("Usage", "mean"),
        Age=(age_col, "first"),
        n_cells=("Usage", "size")
    )
    .reset_index()
)

df_donor["Age"] = pd.to_numeric(df_donor["Age"], errors="coerce")
df_donor = df_donor.dropna(subset=["Age"]).copy()

# transform + scale age
df_donor["Usage_log1p"] = np.log1p(df_donor["Usage"])
df_donor["Age_z"] = (df_donor["Age"] - df_donor["Age"].mean()) / df_donor["Age"].std(ddof=0)

# standardize strings
df_donor[life_col] = df_donor[life_col].astype(str)
df_donor[sex_col] = df_donor[sex_col].astype(str)
df_donor[stim_col] = df_donor[stim_col].astype(str)

print("Donor-level table shape:", df_donor.shape)
print(df_donor.head())

# =========================================================
# 3. FIT ONE INTERACTION MODEL PER PROGRAM x STIMULATION
#    Usage_log1p ~ Lifestyle * Sex + Age_z
#
#    Reference levels:
#      Lifestyle: RURAL
#      Sex: Female
#
#    Extract simple effects from SAME model:
#      A) Urban vs Rural within Female
#      B) Urban vs Rural within Male
#      C) Male vs Female within Rural
#      D) Male vs Female within Urban
# =========================================================
results_simple = []

for program in sorted(df_donor["Program"].unique()):
    for stim in sorted(df_donor[stim_col].unique()):
        sub = df_donor[
            (df_donor["Program"] == program) &
            (df_donor[stim_col] == stim)
        ].copy()

        sub = sub.dropna(subset=["Usage_log1p", "Age_z", life_col, sex_col])

        lifestyles_present = set(sub[life_col].unique())
        sexes_present = set(sub[sex_col].unique())

        if not set(life_levels).issubset(lifestyles_present):
            continue
        if not set(sex_levels).issubset(sexes_present):
            continue
        if sub.shape[0] < 10:
            continue

        sub[life_col] = pd.Categorical(sub[life_col], categories=life_levels, ordered=True)
        sub[sex_col] = pd.Categorical(sub[sex_col], categories=sex_levels, ordered=True)

        model = smf.ols(
            "Usage_log1p ~ C(Lifestyle) * C(Sex) + Age_z",
            data=sub
        ).fit()

        params = model.params
        cov = model.cov_params()
        names = list(params.index)

        term_L = "C(Lifestyle)[T.URBAN]"
        term_S = "C(Sex)[T.Male]"
        term_I = "C(Lifestyle)[T.URBAN]:C(Sex)[T.Male]"

        def linear_combo(term_weights):
            L = np.zeros(len(names))
            for term, weight in term_weights.items():
                if term in names:
                    L[names.index(term)] = weight

            est = float(np.dot(L, params.values))
            se = float(np.sqrt(np.dot(L, np.dot(cov.values, L))))
            tval = est / se if se > 0 else np.nan
            pval = float(model.t_test(L).pvalue)
            return est, se, tval, pval

        # A) Urban vs Rural within Female
        est, se, tval, pval = linear_combo({term_L: 1})
        results_simple.append({
            "Program": program,
            "Stimulation": stim,
            "comparison_type": "Lifestyle_within_Sex",
            "group": "Female",
            "comparison": "Urban_vs_Rural",
            "coef": est,
            "se": se,
            "t": tval,
            "p": pval
        })

        # B) Urban vs Rural within Male
        est, se, tval, pval = linear_combo({term_L: 1, term_I: 1})
        results_simple.append({
            "Program": program,
            "Stimulation": stim,
            "comparison_type": "Lifestyle_within_Sex",
            "group": "Male",
            "comparison": "Urban_vs_Rural",
            "coef": est,
            "se": se,
            "t": tval,
            "p": pval
        })

        # C) Male vs Female within Rural
        est, se, tval, pval = linear_combo({term_S: 1})
        results_simple.append({
            "Program": program,
            "Stimulation": stim,
            "comparison_type": "Sex_within_Lifestyle",
            "group": "RURAL",
            "comparison": "Male_vs_Female",
            "coef": est,
            "se": se,
            "t": tval,
            "p": pval
        })

        # D) Male vs Female within Urban
        est, se, tval, pval = linear_combo({term_S: 1, term_I: 1})
        results_simple.append({
            "Program": program,
            "Stimulation": stim,
            "comparison_type": "Sex_within_Lifestyle",
            "group": "URBAN",
            "comparison": "Male_vs_Female",
            "coef": est,
            "se": se,
            "t": tval,
            "p": pval
        })

res_simple = pd.DataFrame(results_simple)

# BH correction separately by plot family
for comp_type in res_simple["comparison_type"].unique():
    mask = res_simple["comparison_type"] == comp_type
    res_simple.loc[mask, "p_adj_BH"] = multipletests(
        res_simple.loc[mask, "p"], method="fdr_bh"
    )[1]

res_simple["significant_BH_0.05"] = res_simple["p_adj_BH"] < 0.05

print("\nSimple effects extracted from interaction model:")
print(res_simple.head(20))

res_lifestyle_from_interaction = res_simple[
    res_simple["comparison_type"] == "Lifestyle_within_Sex"
].copy()

res_sex_from_interaction = res_simple[
    res_simple["comparison_type"] == "Sex_within_Lifestyle"
].copy()

# =========================================================
# 4. HELPER FUNCTIONS
# =========================================================
def format_p(p):
    if pd.isna(p):
        return ""
    if p < 1e-3:
        return f"FDR={p:.1e}"
    return f"FDR={p:.3f}"

def make_axes_grid(n_programs, n_rows=2, figsize_per_col=4.2, figsize_per_row=3.6):
    n_cols = math.ceil(n_programs / n_rows)
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(n_cols * figsize_per_col, n_rows * figsize_per_row),
        squeeze=False,
        sharey=False
    )
    return fig, axes, n_cols

# =========================================================
# 5. PLOT A
#    SINGLE-CELL BOXPLOTS:
#    Lifestyle differences within Stimulation x Sex
#
#    Layout:
#      rows = Sex (Female, Male)
#      columns = Program
# =========================================================
program_order = sorted(df_long["Program"].unique())
stim_order = sorted(df_long[stim_col].unique())

fig1, axes1 = plt.subplots(
    2, len(program_order),
    figsize=(4.3 * len(program_order), 7.0),
    squeeze=False,
    sharey=False
)

for r, sex in enumerate(sex_levels):
    for c, program in enumerate(program_order):
        ax = axes1[r, c]

        sub_plot = df_long[
            (df_long[sex_col] == sex) &
            (df_long["Program"] == program)
        ].copy()

        if sub_plot.empty:
            ax.set_visible(False)
            continue

        sns.boxplot(
            data=sub_plot,
            x=stim_col,
            y="Usage",
            hue=life_col,
            ax=ax,
            order=stim_order,
            hue_order=life_levels,
            showfliers=False
        )

        # remove per-axis legend
        if ax.get_legend() is not None:
            ax.get_legend().remove()

        ymax = sub_plot["Usage"].max()
        ymin = sub_plot["Usage"].min()
        yrange = ymax - ymin if ymax > ymin else 0.05

        for i, stim in enumerate(stim_order):
            row = res_lifestyle_from_interaction[
                (res_lifestyle_from_interaction["Program"] == program) &
                (res_lifestyle_from_interaction["Stimulation"] == stim) &
                (res_lifestyle_from_interaction["group"] == sex)
            ]
            if row.empty:
                continue

            label = format_p(row["p_adj_BH"].iloc[0])
            ax.text(
                i, ymax + 0.08 * yrange,
                label,
                ha="center", va="bottom",
                fontsize=8
            )

        ax.set_ylim(ymin, ymax + 0.22 * yrange)
        ax.set_title(f"{program}\n{sex}")
        ax.set_xlabel("")
        if c == 0:
            ax.set_ylabel("Usage")
        else:
            ax.set_ylabel("")

# one shared legend
handles, labels = axes1[0, 0].get_legend_handles_labels()
fig1.legend(handles, labels, title="Lifestyle", loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))
fig1.suptitle("Single-cell usage: Lifestyle differences within Stimulation × Sex", y=1.08)
plt.tight_layout()
plt.show()

# =========================================================
# 6. PLOT B
#    SINGLE-CELL BOXPLOTS:
#    Sex differences within Stimulation x Lifestyle
#
#    Layout:
#      rows = Lifestyle (RURAL, URBAN)
#      columns = Program
# =========================================================
fig2, axes2 = plt.subplots(
    2, len(program_order),
    figsize=(4.3 * len(program_order), 7.0),
    squeeze=False,
    sharey=False
)

for r, lifestyle in enumerate(life_levels):
    for c, program in enumerate(program_order):
        ax = axes2[r, c]

        sub_plot = df_long[
            (df_long[life_col] == lifestyle) &
            (df_long["Program"] == program)
        ].copy()

        if sub_plot.empty:
            ax.set_visible(False)
            continue

        sns.boxplot(
            data=sub_plot,
            x=stim_col,
            y="Usage",
            hue=sex_col,
            ax=ax,
            order=stim_order,
            hue_order=sex_levels,
            showfliers=False
        )

        if ax.get_legend() is not None:
            ax.get_legend().remove()

        ymax = sub_plot["Usage"].max()
        ymin = sub_plot["Usage"].min()
        yrange = ymax - ymin if ymax > ymin else 0.05

        for i, stim in enumerate(stim_order):
            row = res_sex_from_interaction[
                (res_sex_from_interaction["Program"] == program) &
                (res_sex_from_interaction["Stimulation"] == stim) &
                (res_sex_from_interaction["group"] == lifestyle)
            ]
            if row.empty:
                continue

            label = format_p(row["p_adj_BH"].iloc[0])
            ax.text(
                i, ymax + 0.08 * yrange,
                label,
                ha="center", va="bottom",
                fontsize=8
            )

        ax.set_ylim(ymin, ymax + 0.22 * yrange)
        ax.set_title(f"{program}\n{lifestyle}")
        ax.set_xlabel("")
        if c == 0:
            ax.set_ylabel("Usage")
        else:
            ax.set_ylabel("")

handles, labels = axes2[0, 0].get_legend_handles_labels()
fig2.legend(handles, labels, title="Sex", loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.02))
fig2.suptitle("Single-cell usage: Sex differences within Stimulation × Lifestyle", y=1.08)
plt.tight_layout()
plt.show()

# =========================================================
# 7. OPTIONAL SAVE TABLES
# =========================================================
res_simple.to_csv("interaction_simple_effects_all.csv", index=False)
res_lifestyle_from_interaction.to_csv("interaction_simple_effects_lifestyle_within_sex.csv", index=False)
res_sex_from_interaction.to_csv("interaction_simple_effects_sex_within_lifestyle.csv", index=False)